# 2D Linear Schrödinger Equation: Young's Double Slit Experiment
## Wave Packet Diffraction through a Potential Barrier

This notebook simulates the classic Young's double-slit experiment. Instead of using source terms at the slits, we model the physical setup directly:
1. An **incoming Gaussian wave packet** travels from the left.
2. It hits a **solid wall** modeled as a high potential barrier $V(x,y)$.
3. The wave reflects off the wall but **diffracts and interferes** as it passes through the two slits (where $V=0$).

---

## 1. The Governing Equation

$$
i\partial_t\psi = -\nabla^2\psi + V(x,y)\psi \quad \implies \quad \partial_t\psi = i\nabla^2\psi - i V(x,y)\psi
$$

*   **Linear Diffraction:** $i\nabla^2\psi$ governs the dispersive propagation.
*   **Potential Barrier:** $-i V(x,y)\psi$ represents the wall. We use a high Gaussian potential for the wall, subtracting Gaussian "holes" for the slits.

Because $V(x,y)$ varies in space, we use the solver's `psiOp` (pseudo-differential operator) to handle the spatially varying symbol.

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Grid and Time ─
# Large domain to prevent boundary reflections and allow wave packet travel
Lx, Ly = 100.0, 50.0
Nx, Ny = 256, 128
Lt, Nt = 20.0, 1000
n_frames = 200

# ── Incoming Wave Packet ──
x_init = -0.0   # Initial center of the wave packet (left side)
w = 4.0          # Width of the wave packet
kx = 1.5         # Momentum in x-direction (controls speed: v = 2*kx)

# ── Wall and Slit Geometry ──
x0 = 0.0         # Wall position (center)
d = 15.0         # Slit separation
wx = 1.5         # Wall "thickness" (Gaussian width in x)
wy = 2.0         # Slit width (Gaussian width in y)
V0 = 500.0       # Height of the potential barrier (must be >> kx^2 to reflect well)

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and Potential Barrier

In [ ]:
t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True)
psi_func  = sp.Function('psi')
psi_field = psi_func(t, x, y)

# ── Potential Barrier V(x,y) ──
# High Gaussian wall at x0, with two Gaussian "holes" (slits) at y = ±d/2
wall = sp.exp(-((x - x0)/wx)**2)
slit1 = sp.exp(-((y - d/2)/wy)**2)
slit2 = sp.exp(-((y + d/2)/wy)**2)

# V(x,y) = V0 * wall * (1 - slit1 - slit2)
# We use (1 - slit1 - slit2) to carve out the holes. 
# To ensure V doesn't go negative, we can just subtract the slits from the full wall.
V_expr = V0 * wall * (1 - slit1 - slit2)

print("Potential Barrier V(x,y):")
print("  V(x,y) =", V_expr)

## 4. Equation with psiOp

In [ ]:
# ∂ψ/t = i∇²ψ - i V(x,y)ψ
# In Fourier space: i∇² → -i(ξ² + η²)
# The potential term -i V(x,y) is spatially varying, so we combine it into the psiOp symbol.

symbol = -sp.I * (xi**2 + eta**2) 

equation = sp.Eq(
    sp.diff(psi_field, t),
    psiOp(symbol - sp.I * V_expr, psi_field)
)

print("Schrödinger Equation with Potential Barrier:")
print("  ∂ψ/∂t = psiOp(-i(ξ² + η²) - i V(x,y), ψ)")

## 5. Initial conditions: Incoming Wave Packet

In [ ]:
def initial_condition_wave_packet(xx, yy):
    """
    Gaussian wave packet centered at x_init, moving to the right with momentum kx.
    ψ(x,y,0) = exp(-((x-x_init)² + y²)/w²) * exp(i kx x)
    """
    envelope = np.exp(-((xx - x_init)**2 + yy**2) / w**2)
    phase = kx * xx
    return envelope * np.exp(1j * phase)

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic', # Periodic is faster; domain is large enough to avoid wrap-around
    initial_condition=initial_condition_wave_packet,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='abs',    # Show |ψ| (probability density / intensity)
    overlay='contour',  # Contours reveal the interference fringes
    mode='imshow',      # 2D raster is ideal for diffraction patterns
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('schrodinger_young_slits_wavepacket.mp4', writer='ffmpeg', fps=20, dpi=150)
print("✅ Saved to schrodinger_young_slits_wavepacket.mp4")